# Отчёт по ML-исследованию для защиты

Этот notebook — компактный презентационный отчёт по ML-блоку Game Intelligence Platform.

Перед запуском рекомендуется выполнить:

```bash
make er-merge-strategy-comparison
make er-graph-analysis
make er-embedding-research
make igdb-matching-analysis
make ml-research-defense
make bayesian-rating
make rag-explanations
make ml-defense-readiness
```

Для полного пересбора `make ml-defense-all`.


In [1]:
import csv
import json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
REPORT_DIR = ROOT / 'data' / 'artifacts' / 'reports'
DEFENSE_DIR = REPORT_DIR / 'ml_research_defense'
ER_DIR = REPORT_DIR / 'entity_resolution'
GRAPH_DIR = REPORT_DIR / 'graph_analysis'
EMBEDDING_DIR = REPORT_DIR / 'embedding_research'
IGDB_DIR = REPORT_DIR / 'igdb_matching'
BAYESIAN_DIR = REPORT_DIR / 'bayesian_rating'
RAG_DIR = REPORT_DIR / 'rag_explanations'
READINESS_DIR = REPORT_DIR / 'ml_defense_readiness'

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def read_csv(path):
    with path.open(encoding='utf-8') as file:
        return list(csv.DictReader(file))

summary = read_json(DEFENSE_DIR / 'ml_research_defense_summary.json')
merge = read_json(ER_DIR / 'merge_strategies' / 'merge_strategy_comparison.json')
graph = read_json(GRAPH_DIR / 'graph_analysis_summary.json')
embedding = read_json(EMBEDDING_DIR / 'embedding_research_summary.json')
igdb = read_json(IGDB_DIR / 'igdb_matching_summary.json')
bayesian_summary = read_json(BAYESIAN_DIR / 'bayesian_rating_summary.json')
rag_summary = read_json(RAG_DIR / 'rag_explanation_summary.json')
readiness_summary = read_json(READINESS_DIR / 'ml_defense_readiness_summary.json')
summary.keys(), merge.keys(), graph.keys(), embedding.keys(), igdb.keys(), bayesian_summary.keys(), rag_summary.keys(), readiness_summary.keys()


(dict_keys(['ablation_study', 'active_learning_candidate_count', 'baseline_counts', 'calibration', 'data_quality', 'defense_demo_case_count', 'existing_er_artifacts', 'grouped_holdout', 'presentation_charts', 'recommendations', 'training_dataset']),
 dict_keys(['candidate_pair_count', 'model_candidate_count', 'model_candidate_threshold', 'risk_case_count', 'source_record_count', 'strategies', 'strategy_count']),
 dict_keys(['candidate_pair_count', 'high_probability_reviewed_negative_count', 'risky_component_count', 'source_record_count', 'strategies', 'strategy_count']),
 dict_keys(['best_f1', 'best_model', 'error_case_count', 'model_count', 'models', 'negative_label_count', 'positive_label_count', 'reviewed_pair_count']),
 dict_keys(['candidate_summary', 'confidence_precision', 'enrichment_coverage', 'high_risk_reviewed_negative_count', 'igdb_pairs', 'igdb_search_candidates', 'igdb_source_games', 'likely_positive_candidate_count', 'rank_precision', 'rawg_anchors', 'reviewed_negative_c

## 1. Базовое состояние данных и ручной разметки

Этот раздел показывает, что в проекте достаточно данных и ручных меток для базовой модели.

In [2]:
summary['baseline_counts']


{'candidate_pairs_by_source': [{'candidate_source': 'external_id',
   'row_count': 6263},
  {'candidate_source': 'igdb_search', 'row_count': 10730},
  {'candidate_source': 'same_normalized_name', 'row_count': 123},
  {'candidate_source': 'same_release_year_and_similar_name',
   'row_count': 204}],
 'dm_counts': [{'object_name': 'canonical_game_sources', 'row_count': 35607},
  {'object_name': 'canonical_games', 'row_count': 20613},
  {'object_name': 'game_recommendations', 'row_count': 147254}],
 'manual_review_labels': [{'review_label': False,
   'review_status': 'reviewed',
   'row_count': 937},
  {'review_label': True, 'review_status': 'reviewed', 'row_count': 1029},
  {'review_label': None, 'review_status': 'skipped', 'row_count': 152}],
 'model_decisions': [{'decision': 'auto_merge', 'row_count': 18},
  {'decision': 'manual_review', 'row_count': 10862},
  {'decision': 'no_merge', 'row_count': 6440}],
 'source_games_by_source': [{'row_count': 10344, 'source': 'igdb'},
  {'row_count'

In [3]:
summary['training_dataset']


{'label_source_counts': [{'len': 1965,
   'training_label_source': 'manual_review'},
  {'len': 5852, 'training_label_source': 'weak_positive'}],
 'negative_count': 936,
 'positive_count': 6881,
 'row_count': 7817}

## 2. Влияние качества данных

Артефакты контроля качества данных объясняют, почему Entity Resolution требует ручной проверки и консервативных threshold-правил.

In [4]:
summary['data_quality']


{'anomaly_preview': [{'anomaly_type': 'possible_dlc_or_edition_by_title',
   'detail': 'Distraint: Deluxe Edition',
   'source': 'igdb',
   'source_game_id': '100761'},
  {'anomaly_type': 'possible_dlc_or_edition_by_title',
   'detail': 'Ticket to Ride: Classic Edition',
   'source': 'igdb',
   'source_game_id': '10745'},
  {'anomaly_type': 'possible_dlc_or_edition_by_title',
   'detail': 'Evoland Legendary Edition',
   'source': 'igdb',
   'source_game_id': '114910'},
  {'anomaly_type': 'possible_dlc_or_edition_by_title',
   'detail': 'Call of Duty: Black Ops 4 - Digital Deluxe Edition',
   'source': 'igdb',
   'source_game_id': '118732'},
  {'anomaly_type': 'possible_dlc_or_edition_by_title',
   'detail': 'Deus Ex: Human Revolution - Complete Edition',
   'source': 'igdb',
   'source_game_id': '118891'},
  {'anomaly_type': 'possible_dlc_or_edition_by_title',
   'detail': 'Sonic & All-Stars Racing Transformed: Metal Sonic & Outrun DLC',
   'source': 'igdb',
   'source_game_id': '11918

## 3. Метрики Entity Resolution

Модель полезна как слой скоринга, ручной проверки и контролируемого гибридного объединения, а не как слепой автоматический canonical merge.

In [5]:
summary['existing_er_artifacts']['v3c_metrics']


{'confusion_matrix': [[331, 43], [230, 2523]],
 'f1': 0.948675,
 'negative_label_count': 936,
 'positive_label_count': 6881,
 'pr_auc': 0.994369,
 'precision': 0.983242,
 'recall': 0.916455,
 'roc_auc': 0.957274,
 'sample_weight_policy': {'manual_review': 1.0,
  'synthetic_negative': 0.5,
  'weak_positive': 0.05},
 'test_manual_review_metrics': {'confusion_matrix': [[331, 43], [8, 407]],
  'f1': 0.94104,
  'label_source': 'manual_review',
  'negative_label_count': 374,
  'positive_label_count': 415,
  'pr_auc': 0.99213,
  'precision': 0.904444,
  'recall': 0.980723,
  'roc_auc': 0.98887,
  'row_count': 789},
 'test_row_count': 3127,
 'training_label_source_counts': [{'len': 1965,
   'training_label_source': 'manual_review'},
  {'len': 5852, 'training_label_source': 'weak_positive'}],
 'training_row_count': 4690}

In [6]:
read_csv(ER_DIR / 'iterations' / 'manual_threshold_eval_v3c.csv')


[{'threshold': '0.50',
  'predicted_positive': '1097',
  'tp': '1005',
  'fp': '92',
  'fn': '24',
  'precision': '0.916135',
  'recall': '0.976676'},
 {'threshold': '0.70',
  'predicted_positive': '1025',
  'tp': '997',
  'fp': '28',
  'fn': '32',
  'precision': '0.972683',
  'recall': '0.968902'},
 {'threshold': '0.90',
  'predicted_positive': '415',
  'tp': '415',
  'fp': '0',
  'fn': '614',
  'precision': '1.000000',
  'recall': '0.403304'},
 {'threshold': '0.95',
  'predicted_positive': '14',
  'tp': '14',
  'fp': '0',
  'fn': '1015',
  'precision': '1.000000',
  'recall': '0.013605'},
 {'threshold': '0.98',
  'predicted_positive': '0',
  'tp': '0',
  'fp': '0',
  'fn': '1029',
  'precision': '',
  'recall': '0.000000'},
 {'threshold': '0.99',
  'predicted_positive': '0',
  'tp': '0',
  'fp': '0',
  'fn': '1029',
  'precision': '',
  'recall': '0.000000'}]

## 4. Исследование важности признаков и калибровки

Ablation study показывает, какие группы признаков наиболее важны. Calibration проверяет, можно ли использовать вероятностные оценки модели для threshold policy.

In [7]:
read_csv(DEFENSE_DIR / 'ablation_study.csv')


[{'confusion_matrix': '[[331, 43], [230, 2523]]',
  'excluded_features': '',
  'f1': '0.948675',
  'feature_count': '12',
  'pr_auc': '0.994369',
  'precision': '0.983242',
  'recall': '0.916455',
  'roc_auc': '0.957274',
  'scenario': 'all_features'},
 {'confusion_matrix': '[[272, 102], [323, 2430]]',
  'excluded_features': 'alias_similarity, name_similarity',
  'f1': '0.919584',
  'feature_count': '10',
  'pr_auc': '0.975892',
  'precision': '0.959716',
  'recall': '0.882673',
  'roc_auc': '0.864896',
  'scenario': 'without_name'},
 {'confusion_matrix': '[[325, 49], [201, 2552]]',
  'excluded_features': 'release_year_diff',
  'f1': '0.953306',
  'feature_count': '11',
  'pr_auc': '0.98601',
  'precision': '0.981161',
  'recall': '0.926989',
  'roc_auc': '0.930256',
  'scenario': 'without_year'},
 {'confusion_matrix': '[[331, 43], [230, 2523]]',
  'excluded_features': 'external_id_exact_match',
  'f1': '0.948675',
  'feature_count': '11',
  'pr_auc': '0.994369',
  'precision': '0.9832

In [8]:
summary['calibration']


{'bins': [{'avg_probability': 0.020499,
   'bin': '0.0-0.1',
   'count': 355,
   'positive_rate': 0.346479},
  {'avg_probability': 0.143112,
   'bin': '0.1-0.2',
   'count': 50,
   'positive_rate': 0.78},
  {'avg_probability': 0.241596,
   'bin': '0.2-0.3',
   'count': 46,
   'positive_rate': 0.5},
  {'avg_probability': 0.336856,
   'bin': '0.3-0.4',
   'count': 60,
   'positive_rate': 0.433333},
  {'avg_probability': 0.455281,
   'bin': '0.4-0.5',
   'count': 50,
   'positive_rate': 0.38},
  {'avg_probability': 0.540228,
   'bin': '0.5-0.6',
   'count': 58,
   'positive_rate': 0.586207},
  {'avg_probability': 0.654138,
   'bin': '0.6-0.7',
   'count': 69,
   'positive_rate': 0.898551},
  {'avg_probability': 0.767384,
   'bin': '0.7-0.8',
   'count': 183,
   'positive_rate': 0.956284},
  {'avg_probability': 0.874214,
   'bin': '0.8-0.9',
   'count': 1044,
   'positive_rate': 0.996169},
  {'avg_probability': 0.918709,
   'bin': '0.9-1.0',
   'count': 1212,
   'positive_rate': 1.0}],
 'b

### Графики

![F1 в исследовании важности признаков](../data/artifacts/reports/ml_research_defense/charts/ablation_f1.svg)

![Калибровка вероятностей](../data/artifacts/reports/ml_research_defense/charts/calibration_bins.svg)

## 5. Сравнение стратегий объединения

Теневое сравнение стратегий показывает, почему автоматическое объединение только по модели требует governance-контроля и не должно напрямую заменять доверенный canonical layer.

In [9]:
merge['strategies']


[{'component_count': 20613,
  'description': 'Current trusted canonical v1: deterministic external-id edges plus reviewed positives.',
  'edge_count': 6012,
  'f1': 0.999029,
  'false_negative': 0,
  'false_positive': 2,
  'max_component_size': 5,
  'model_edge_count': 0,
  'multi_source_component_count': 6331,
  'precision': 0.99806,
  'recall': 1.0,
  'reviewed_pair_count': 1966,
  'same_source_duplicate_components': 29,
  'same_source_duplicate_links': 31,
  'strategy': 'canonical_v1_trusted',
  'true_negative': 935,
  'true_positive': 1029,
  'trusted_edge_count': 6012},
 {'component_count': 26606,
  'description': 'Model-only auto merge at probability >= 0.95.',
  'edge_count': 18,
  'f1': 0.026846,
  'false_negative': 1015,
  'false_positive': 0,
  'max_component_size': 3,
  'model_edge_count': 18,
  'multi_source_component_count': 4834,
  'precision': 1.0,
  'recall': 0.013605,
  'reviewed_pair_count': 1966,
  'same_source_duplicate_components': 22,
  'same_source_duplicate_link

![F1 по стратегиям объединения](../data/artifacts/reports/ml_research_defense/charts/merge_strategy_f1.svg)

## 6. Анализ ER-графа рисков

Анализ графа рисков показывает транзитивный риск merge-ошибок: качество на уровне отдельных пар может выглядеть сильным, но политика объединения всё равно может создавать компоненты с дубликатами из одного источника или рискованные кластеры.

In [10]:
graph['strategies']


[{'component_count': 20613,
  'description': 'Current trusted canonical v1: deterministic external-id edges plus reviewed positives.',
  'edge_count': 6012,
  'max_component_size': 5,
  'model_edge_count': 0,
  'multi_source_component_count': 6331,
  'risky_component_count': 29,
  'same_source_duplicate_components': 29,
  'same_source_duplicate_links': 31,
  'strategy': 'canonical_v1_trusted',
  'trusted_edge_count': 6012},
 {'component_count': 26606,
  'description': 'Model-only auto merge at probability >= 0.95.',
  'edge_count': 18,
  'max_component_size': 3,
  'model_edge_count': 18,
  'multi_source_component_count': 4834,
  'risky_component_count': 22,
  'same_source_duplicate_components': 22,
  'same_source_duplicate_links': 22,
  'strategy': 'model_auto_095',
  'trusted_edge_count': 0},
 {'component_count': 23441,
  'description': 'Model-only auto merge at probability >= 0.90.',
  'edge_count': 3183,
  'max_component_size': 5,
  'model_edge_count': 3183,
  'multi_source_componen

In [11]:
read_csv(GRAPH_DIR / 'risky_components.csv')[:20]


[{'component_id': '55',
  'component_label': 'APB Reloaded | APB Reloaded | APB Reloaded | APB Reloaded | All Points Bulletin',
  'component_size': '5',
  'manual_negative_pairs_in_component': '1',
  'model_edge_count': '0',
  'risk_score': '110',
  'same_source_duplicate_links': '1',
  'same_source_duplicate_sources': 'wikidata:2',
  'selected_edge_count': '2',
  'source_count': '4',
  'source_distribution': 'igdb:1, rawg:1, steam:1, wikidata:2',
  'source_keys_preview': 'igdb:1014; rawg:51; steam:113400; wikidata:Q126010830; wikidata:Q296101',
  'strategy': 'canonical_v1_trusted',
  'trusted_edge_count': '2'},
 {'component_id': '15706',
  'component_label': 'Under Night In-Birth Exe:Late | UNDER NIGHT IN-BIRTH Exe:Late | Under Night In-Birth Exe:Late | Under Night In-Birth',
  'component_size': '4',
  'manual_negative_pairs_in_component': '1',
  'model_edge_count': '0',
  'risk_score': '110',
  'same_source_duplicate_links': '1',
  'same_source_duplicate_sources': 'wikidata:2',
  'se

In [12]:
read_csv(GRAPH_DIR / 'high_probability_reviewed_negatives.csv')[:20]


[{'candidate_source': 'same_release_year_and_similar_name',
  'name_a': 'The Jackbox Party Pack 4',
  'name_b': 'The Jackbox Party Pack',
  'name_similarity': '0.956522',
  'pair_id': '1403da61-a6fe-4a48-811e-64e22108984a',
  'release_year_a': '2017',
  'release_year_b': '2017',
  'release_year_diff': '0',
  'review_notes': 'v2 review queue',
  'same_game_probability': '0.838165',
  'source_a': 'rawg',
  'source_b': 'wikidata',
  'source_id_a': '47118',
  'source_id_b': 'Q54802474',
  'trusted_canonical_edge': 'False'},
 {'candidate_source': 'same_release_year_and_similar_name',
  'name_a': 'BioShock 2 Remastered',
  'name_b': 'BioShock Remastered',
  'name_similarity': '0.950000',
  'pair_id': '5169e74a-0b47-46bd-80a4-f1390816cb92',
  'release_year_a': '2016',
  'release_year_b': '2016',
  'release_year_diff': '0',
  'review_notes': 'v2 review queue',
  'same_game_probability': '0.829243',
  'source_a': 'rawg',
  'source_b': 'wikidata',
  'source_id_a': '11142',
  'source_id_b': 'Q115

![Дублирующие связи из одного источника](../data/artifacts/reports/graph_analysis/same_source_duplicate_links.svg)

## 7. Лёгкое исследование эмбеддингов названий

Локальный TF-IDF/SVD эксперимент сравнивает векторную близость названий с fuzzy-сходством названий до добавления более тяжёлых нейросетевых эмбеддингов.

In [13]:
embedding


{'best_f1': 0.974026,
 'best_model': 'combined_name_embedding_year',
 'error_case_count': 35,
 'model_count': 6,
 'models': [{'brier_score': 0.022386,
   'f1': 0.974026,
   'features': 'name_similarity, char_tfidf_cosine, word_tfidf_cosine, title_svd_cosine, release_year_abs_diff, same_release_year',
   'model': 'combined_name_embedding_year',
   'negative_test_rows': 281,
   'positive_predictions': 307,
   'positive_test_rows': 309,
   'pr_auc': 0.99347,
   'precision': 0.977199,
   'recall': 0.970874,
   'roc_auc': 0.989232,
   'test_rows': 590,
   'threshold': 0.5,
   'train_rows': 1376},
  {'brier_score': 0.045359,
   'f1': 0.953895,
   'features': 'word_tfidf_cosine',
   'model': 'word_tfidf_only',
   'negative_test_rows': 281,
   'positive_predictions': 320,
   'positive_test_rows': 309,
   'pr_auc': 0.953786,
   'precision': 0.9375,
   'recall': 0.970874,
   'roc_auc': 0.963883,
   'test_rows': 590,
   'threshold': 0.5,
   'train_rows': 1376},
  {'brier_score': 0.044052,
   'f1'

In [14]:
read_csv(EMBEDDING_DIR / 'embedding_model_comparison.csv')


[{'brier_score': '0.022386',
  'f1': '0.974026',
  'features': 'name_similarity, char_tfidf_cosine, word_tfidf_cosine, title_svd_cosine, release_year_abs_diff, same_release_year',
  'model': 'combined_name_embedding_year',
  'negative_test_rows': '281',
  'positive_predictions': '307',
  'positive_test_rows': '309',
  'pr_auc': '0.99347',
  'precision': '0.977199',
  'recall': '0.970874',
  'roc_auc': '0.989232',
  'test_rows': '590',
  'threshold': '0.5',
  'train_rows': '1376'},
 {'brier_score': '0.045359',
  'f1': '0.953895',
  'features': 'word_tfidf_cosine',
  'model': 'word_tfidf_only',
  'negative_test_rows': '281',
  'positive_predictions': '320',
  'positive_test_rows': '309',
  'pr_auc': '0.953786',
  'precision': '0.9375',
  'recall': '0.970874',
  'roc_auc': '0.963883',
  'test_rows': '590',
  'threshold': '0.5',
  'train_rows': '1376'},
 {'brier_score': '0.044052',
  'f1': '0.952381',
  'features': 'char_tfidf_cosine, word_tfidf_cosine, title_svd_cosine',
  'model': 'embed

In [15]:
read_csv(EMBEDDING_DIR / 'embedding_error_cases.csv')[:20]


[{'candidate_source': 'same_release_year_and_similar_name',
  'case_type': 'reviewed_negative_high_embedding_probability',
  'char_tfidf_cosine': '0.990761',
  'embedding_same_game_probability': '0.880734',
  'manual_label': '0',
  'name_a': 'The Jackbox Party Pack 4',
  'name_b': 'The Jackbox Party Pack',
  'name_similarity': '0.956522',
  'pair_id': '1403da61-a6fe-4a48-811e-64e22108984a',
  'release_year_a': '2017',
  'release_year_abs_diff': '0.0',
  'release_year_b': '2017',
  'source_a': 'rawg',
  'source_b': 'wikidata',
  'source_id_a': '47118',
  'source_id_b': 'Q54802474',
  'title_svd_cosine': '0.905145',
  'word_tfidf_cosine': '0.880117'},
 {'candidate_source': 'same_release_year_and_similar_name',
  'case_type': 'reviewed_negative_high_embedding_probability',
  'char_tfidf_cosine': '0.994141',
  'embedding_same_game_probability': '0.844516',
  'manual_label': '0',
  'name_a': 'Labyronia RPG',
  'name_b': 'Labyronia RPG 2',
  'name_similarity': '0.928571',
  'pair_id': 'd6255

![F1 embedding-моделей](../data/artifacts/reports/embedding_research/embedding_model_f1.svg)

## 8. Анализ сопоставления через IGDB

Раздел оценивает IGDB как источник поисковых ER-кандидатов: качество retrieval rank, точность проверенных пар, рискованные отрицательные примеры и покрытие enrichment-полей.

In [16]:
igdb


{'candidate_summary': {'query_strategy_summary': [{'anchor_count': 5600,
    'avg_confidence': 0.843913,
    'avg_search_rank': 1.89754,
    'candidate_count': 10326,
    'query_strategy': 'source_name'},
   {'anchor_count': 197,
    'avg_confidence': 0.814975,
    'avg_search_rank': 2.126238,
    'candidate_count': 404,
    'query_strategy': 'rawg_slug'}],
  'rank_distribution': [{'anchor_count': 5686,
    'avg_confidence': 0.941794,
    'candidate_count': 5770,
    'search_rank': 1},
   {'anchor_count': 2152,
    'avg_confidence': 0.824954,
    'candidate_count': 2178,
    'search_rank': 2},
   {'anchor_count': 1385,
    'avg_confidence': 0.723407,
    'candidate_count': 1397,
    'search_rank': 3},
   {'anchor_count': 777,
    'avg_confidence': 0.62256,
    'candidate_count': 789,
    'search_rank': 4},
   {'anchor_count': 591,
    'avg_confidence': 0.521477,
    'candidate_count': 596,
    'search_rank': 5}],
  'source_summary': [{'anchor_count': 5686,
    'avg_confidence': 0.84282

In [17]:
read_csv(IGDB_DIR / 'igdb_review_precision_by_rank.csv')


[{'bucket_name': 'igdb_rank_bucket',
  'bucket_value': '1',
  'reviewed_count': '968',
  'reviewed_negative_count': '69',
  'reviewed_positive_count': '899',
  'reviewed_precision': '0.928719'},
 {'bucket_name': 'igdb_rank_bucket',
  'bucket_value': '2-3',
  'reviewed_count': '439',
  'reviewed_negative_count': '416',
  'reviewed_positive_count': '23',
  'reviewed_precision': '0.052392'},
 {'bucket_name': 'igdb_rank_bucket',
  'bucket_value': '4-5',
  'reviewed_count': '230',
  'reviewed_negative_count': '227',
  'reviewed_positive_count': '3',
  'reviewed_precision': '0.013043'}]

In [18]:
read_csv(IGDB_DIR / 'igdb_enrichment_coverage.csv')


[{'coverage_rate': '0.644142',
  'feature': 'aliases',
  'game_count': '6663',
  'igdb_game_count': '10344'},
 {'coverage_rate': '0.854698',
  'feature': 'companies',
  'game_count': '8841',
  'igdb_game_count': '10344'},
 {'coverage_rate': '0.975058',
  'feature': 'descriptions',
  'game_count': '10086',
  'igdb_game_count': '10344'},
 {'coverage_rate': '0.947989',
  'feature': 'genres',
  'game_count': '9806',
  'igdb_game_count': '10344'},
 {'coverage_rate': '0.996906',
  'feature': 'platforms',
  'game_count': '10312',
  'igdb_game_count': '10344'},
 {'coverage_rate': '0.595514',
  'feature': 'ratings',
  'game_count': '6160',
  'igdb_game_count': '10344'},
 {'coverage_rate': '0.614076',
  'feature': 'tags',
  'game_count': '6352',
  'igdb_game_count': '10344'},
 {'coverage_rate': '0.818832',
  'feature': 'themes',
  'game_count': '8470',
  'igdb_game_count': '10344'},
 {'coverage_rate': '0.91628',
  'feature': 'urls',
  'game_count': '9478',
  'igdb_game_count': '10344'}]

In [19]:
read_csv(IGDB_DIR / 'igdb_high_risk_reviewed_negatives.csv')[:20]


[{'alias_similarity': '',
  'case_type': 'reviewed_negative_high_model_score',
  'igdb_query_strategy': 'source_name',
  'igdb_search_confidence': '0.7500',
  'igdb_search_rank': '3',
  'model_decision': 'manual_review',
  'name_a': 'Call of Duty: Modern Warfare 2',
  'name_b': 'Call of Duty: Modern Warfare 2 - Veteran Package',
  'name_similarity': '0.783784',
  'pair_id': 'f834ac88-d448-438f-9545-d564315fd29e',
  'release_year_a': '2009',
  'release_year_b': '2009',
  'release_year_diff': '0',
  'review_label': 'False',
  'review_notes': 'different game; edition',
  'review_status': 'reviewed',
  'same_game_probability': '0.735559',
  'selection_strategy': 'igdb_borderline',
  'source_a': 'rawg',
  'source_b': 'igdb',
  'source_id_a': '4527',
  'source_id_b': '154707'}]

![Распределение IGDB rank](../data/artifacts/reports/igdb_matching/igdb_rank_distribution.svg)

![Покрытие IGDB enrichment](../data/artifacts/reports/igdb_matching/igdb_enrichment_coverage.svg)

## 9. Активное обучение и примеры рекомендаций

Какие пары модель предложила бы проверить следующими и как базовые рекомендации объясняют общие признаки.

In [20]:
read_csv(DEFENSE_DIR / 'active_learning_candidates.csv')[:20]


[{'active_learning_bucket': 'high_uncertainty',
  'candidate_source': 'igdb_search',
  'name_a': 'Super Time Force',
  'name_b': 'Super Time Force Ultra',
  'name_similarity': '0.842105',
  'pair_id': 'bb95a4ed-84df-4020-a8a8-b2405979fde8',
  'release_year_a': '2014',
  'release_year_b': '2014',
  'release_year_diff': '0',
  'review_label': '',
  'review_status': '',
  'same_game_probability': '0.500549',
  'source_a': 'rawg',
  'source_b': 'igdb',
  'source_id_a': '1898',
  'source_id_b': '8879'},
 {'active_learning_bucket': 'high_uncertainty',
  'candidate_source': 'igdb_search',
  'name_a': 'Quake III Arena',
  'name_b': 'Quake III: Team Arena',
  'name_similarity': '0.857143',
  'pair_id': '054911ff-f8bd-485a-8c8a-741f29c61f59',
  'release_year_a': '1999',
  'release_year_b': '2000',
  'release_year_diff': '1',
  'review_label': '',
  'review_status': '',
  'same_game_probability': '0.500612',
  'source_a': 'rawg',
  'source_b': 'igdb',
  'source_id_a': '54718',
  'source_id_b': '6

In [21]:
read_csv(DEFENSE_DIR / 'recommendation_examples.csv')[:20]


[{'explanation_factors_json': '{"shared_features": ["genre:sport", "developer:hb studios", "publisher:ea sports", "platform:playstation portable", "decade:2010s"], "source_feature_count": 5, "candidate_feature_count": 5}',
  'rank': '1',
  'recommended_game': 'FIFA Soccer 11',
  'score': '1.000000',
  'seed_game': '2010 FIFA World Cup South Africa'},
 {'explanation_factors_json': '{"shared_features": ["genre:sport", "publisher:ea sports", "platform:ios", "decade:2010s"], "source_feature_count": 4, "candidate_feature_count": 4}',
  'rank': '1',
  'recommended_game': 'Madden NFL 11',
  'score': '1.000000',
  'seed_game': '2010 FIFA World Cup South Africa'},
 {'explanation_factors_json': '{"shared_features": ["genre:sport", "publisher:ea sports", "platform:ios", "decade:2010s"], "source_feature_count": 4, "candidate_feature_count": 4}',
  'rank': '2',
  'recommended_game': 'Madden NFL 13 Social',
  'score': '1.000000',
  'seed_game': '2010 FIFA World Cup South Africa'},
 {'explanation_fac

## 10. Примеры

Успешные объединения, отклонённые рискованные совпадения, пары для активного обучения и примеры рекомендаций.

In [22]:
read_csv(DEFENSE_DIR / 'defense_demo_cases.csv')


[{'case_type': 'active_learning_candidate',
  'interpretation': 'Unlabeled high-uncertainty pair for the next manual-review batch',
  'item_a': 'Remnant 2',
  'item_b': 'Remnant II',
  'score': '0.500723'},
 {'case_type': 'active_learning_candidate',
  'interpretation': 'Unlabeled high-uncertainty pair for the next manual-review batch',
  'item_a': 'Quake III Arena',
  'item_b': 'Quake III: Team Arena',
  'score': '0.500612'},
 {'case_type': 'active_learning_candidate',
  'interpretation': 'Unlabeled high-uncertainty pair for the next manual-review batch',
  'item_a': 'Super Time Force',
  'item_b': 'Super Time Force Ultra',
  'score': '0.500549'},
 {'case_type': 'rejected_risky_match',
  'interpretation': 'Reviewed negative: similar title but different game/entity',
  'item_a': 'The Jackbox Party Pack 4',
  'item_b': 'The Jackbox Party Pack',
  'score': '0.838165'},
 {'case_type': 'rejected_risky_match',
  'interpretation': 'Reviewed negative: similar title but different game/entity',

In [23]:
read_csv(DEFENSE_DIR / 'recommendation_score_distribution.csv')


[{'row_count': '14837', 'score_bucket': '0.1-0.2'},
 {'row_count': '40417', 'score_bucket': '0.2-0.3'},
 {'row_count': '40976', 'score_bucket': '0.3-0.4'},
 {'row_count': '26171', 'score_bucket': '0.4-0.5'},
 {'row_count': '11137', 'score_bucket': '0.5-0.6'},
 {'row_count': '5574', 'score_bucket': '0.6-0.7'},
 {'row_count': '3115', 'score_bucket': '0.7-0.8'},
 {'row_count': '2245', 'score_bucket': '0.8-0.9'},
 {'row_count': '1173', 'score_bucket': '0.9-1.0'},
 {'row_count': '1609', 'score_bucket': '1.0'}]

![Распределение recommendation score](../data/artifacts/reports/ml_research_defense/charts/recommendation_score_distribution.svg)

## 11. Grounded RAG-like explanations

Раздел показывает русскоязычные объяснения, сгенерированные только из рассчитанных фактов. LLM/RAG-like слой не принимает решения по совпадениям или рекомендациям.

In [24]:
rag_summary


{'fact_card_count': 50,
 'grounding_policy': 'Explanations use computed features, model predictions, manual labels, canonical facts and recommendation shared features only.',
 'language': 'ru',
 'match_explanation_count': 25,
 'recommendation_explanation_count': 25}

In [25]:
read_csv(RAG_DIR / 'match_explanation_examples.csv')[:10]


[{'case_type': 'reviewed_positive',
  'explanation_type': 'match',
  'grounded_explanation_ru': 'Пара `Call of Duty: Modern Warfare (2019) (rawg:323065)` и `Call of Duty: Modern Warfare - Season One (igdb:129865)`: модель считает пару кандидатом на отказ от объединения (50.0%). Основание: сходство названий: 80.0%; сходство alias: нет данных; разница годов выпуска: 0; пересечение разработчиков: 20.0%; пересечение издателей: 0.0%; пересечение платформ: 50.0%; пересечение жанров: 50.0%. Важно: объяснение основано только на рассчитанных признаках, прогнозе модели и ручной разметке, если она есть.',
  'model_decision': 'no_merge',
  'pair_id': 'd2c41837-315c-4414-8ca5-1e8d0fc8568c',
  'review_label': 'True',
  'review_status': 'reviewed',
  'same_game_probability': '0.500054',
  'subject': 'Call of Duty: Modern Warfare (2019) <> Call of Duty: Modern Warfare - Season One'},
 {'case_type': 'reviewed_positive',
  'explanation_type': 'match',
  'grounded_explanation_ru': 'Пара `Spider-Man (2000

In [26]:
read_csv(RAG_DIR / 'recommendation_explanation_examples.csv')[:10]


[{'algorithm': 'content_jaccard_v1',
  'explanation_type': 'recommendation',
  'grounded_explanation_ru': 'Для игры `2010 FIFA World Cup South Africa` рекомендация `FIFA Soccer 11` объясняется совпадением контента. Основание: общий признак `жанр: sport`; общий признак `разработчик: hb studios`; общий признак `издатель: ea sports`; общий признак `платформа: playstation portable`; общий признак `десятилетие: 2010s`; годы выпуска: 2010 и 2010; content score: 100.0%. Это content-based объяснение без user interactions и без LLM-решений.',
  'rank': '1',
  'recommended_game': 'FIFA Soccer 11',
  'score': '1.000000',
  'seed_game': '2010 FIFA World Cup South Africa'},
 {'algorithm': 'content_jaccard_v1',
  'explanation_type': 'recommendation',
  'grounded_explanation_ru': 'Для игры `2010 FIFA World Cup South Africa` рекомендация `Madden NFL 11` объясняется совпадением контента. Основание: общий признак `жанр: sport`; общий признак `издатель: ea sports`; общий признак `платформа: ios`; общий п

In [27]:
read_csv(RAG_DIR / 'grounded_fact_cards.csv')[:10]


[{'canonical_game_id': '8a2f0c1f-e4e9-5511-be70-db528dc68a40',
  'canonical_name': 'Amnesia: The Dark Descent',
  'description_source_count': '4',
  'release_year': '2010',
  'source_count': '5',
  'sources': 'igdb, rawg, steam, wikidata, wikipedia'},
 {'canonical_game_id': '5cc40a0b-72ee-5257-9cf0-5d1419a84804',
  'canonical_name': 'Apex Legends',
  'description_source_count': '4',
  'release_year': '2019',
  'source_count': '5',
  'sources': 'igdb, rawg, steam, wikidata, wikipedia'},
 {'canonical_game_id': '9b191c1b-6e20-597e-af15-00a1f8d3403e',
  'canonical_name': "Baldur's Gate III",
  'description_source_count': '4',
  'release_year': '2023',
  'source_count': '5',
  'sources': 'igdb, rawg, steam, wikidata, wikipedia'},
 {'canonical_game_id': '62eab14e-a81a-5f69-b5cd-dcfe0a1a4d60',
  'canonical_name': 'BattleBlock Theater',
  'description_source_count': '4',
  'release_year': '2013',
  'source_count': '5',
  'sources': 'igdb, rawg, steam, wikidata, wikipedia'},
 {'canonical_game_i

## 12. Анализ Bayesian rating

Bayesian rating — дополнительный статистический блок: он сравнивает наивные рейтинги источников с рейтингами, скорректированными по числу голосов, и показывает, почему игры с малым числом голосов нужно сдвигать к global mean.

In [28]:
bayesian_summary


{'canonical_game_count': 14468,
 'global_weighted_mean': 78.527138,
 'prior_votes': 50.0,
 'rating_input_count': 24344,
 'total_vote_count': 2803272}

In [29]:
read_csv(BAYESIAN_DIR / 'canonical_bayesian_ratings.csv')[:20]


[{'bayesian_rating': '96.497586',
  'canonical_game_id': '55cbde69-db6a-5fe4-8906-3b80667aa245',
  'canonical_name': 'The Witcher 3 Wild Hunt - Complete Edition',
  'naive_weighted_rating': '96.96266',
  'rating_source_count': '3',
  'rating_types': 'igdb_rating, igdb_total_rating, rawg_rating',
  'release_year': '2016',
  'shrinkage_delta': '-0.465074',
  'vote_count': '1932'},
 {'bayesian_rating': '94.894766',
  'canonical_game_id': 'bb272a2b-28b1-5735-8f62-1ef5d28b4288',
  'canonical_name': 'The Witcher 3: Wild Hunt – Blood and Wine',
  'naive_weighted_rating': '96.2',
  'rating_source_count': '1',
  'rating_types': 'rawg_rating',
  'release_year': '2016',
  'shrinkage_delta': '-1.305234',
  'vote_count': '627'},
 {'bayesian_rating': '94.398386',
  'canonical_game_id': 'b3b02214-0f6c-5eaf-885a-d87b95bd3d46',
  'canonical_name': 'The Witcher 3: Wild Hunt - Blood and Wine',
  'naive_weighted_rating': '95.23196',
  'rating_source_count': '3',
  'rating_types': 'igdb_aggregated_rating, 

In [30]:
read_csv(BAYESIAN_DIR / 'low_vote_shrinkage_examples.csv')[:20]


[{'bayesian_rating': '75.891478',
  'canonical_game_id': '08c65ce5-d0bc-5b30-adf2-53ebc4aa1300',
  'canonical_name': 'Monumental',
  'naive_weighted_rating': '10.0',
  'rating_source_count': '2',
  'rating_types': 'igdb_aggregated_rating, igdb_total_rating',
  'release_year': '2016',
  'shrinkage_delta': '65.891478',
  'vote_count': '2'},
 {'bayesian_rating': '74.191794',
  'canonical_game_id': 'dc9ddbb7-eed1-5baa-badd-c8b07c747014',
  'canonical_name': 'Agricultural Simulator 2013: Steam Edition',
  'naive_weighted_rating': '20.0',
  'rating_source_count': '2',
  'rating_types': 'igdb_rating, igdb_total_rating',
  'release_year': '2013',
  'shrinkage_delta': '54.191794',
  'vote_count': '4'},
 {'bayesian_rating': '72.256373',
  'canonical_game_id': '831f1baa-a40c-5cc7-9467-ce604e0703a7',
  'canonical_name': '//SNOWFLAKE TATTOO//',
  'naive_weighted_rating': '20.0',
  'rating_source_count': '1',
  'rating_types': 'rawg_rating',
  'release_year': '2015',
  'shrinkage_delta': '52.256373'

![Лучшие Bayesian ratings](../data/artifacts/reports/bayesian_rating/top_bayesian_ratings.svg)

## 13. Финальный shapshot

Проверка, что все обязательные исследовательские артефакты и ключевые метрики доступны

In [31]:
readiness_summary


{'demo_step_count': 9,
 'metric_count': 9,
 'missing_metric_count': 0,
 'missing_required_count': 0,
 'overall_status': 'ready',
 'required_artifact_count': 10}

In [32]:
read_csv(READINESS_DIR / 'ml_defense_metric_snapshot.csv')


[{'interpretation': 'Total normalized source records in staging.',
  'metric': 'source_records',
  'section': 'Data',
  'value': '31462'},
 {'interpretation': 'Entity-resolution candidate pairs.',
  'metric': 'candidate_pairs',
  'section': 'ER',
  'value': '17320'},
 {'interpretation': 'Manual labels available for supervised ER evaluation.',
  'metric': 'manual_reviewed_labels',
  'section': 'ER',
  'value': '1966'},
 {'interpretation': 'Weighted Logistic Regression baseline F1.',
  'metric': 'v3c_f1',
  'section': 'ER',
  'value': '0.948675'},
 {'interpretation': 'Exported risky graph components across merge strategies.',
  'metric': 'risky_components',
  'section': 'Graph',
  'value': '248'},
 {'interpretation': 'Best F1 from local TF-IDF/SVD title-vector research lane.',
  'metric': 'best_lightweight_embedding_f1',
  'section': 'Embeddings',
  'value': '0.974026'},
 {'interpretation': 'Rank-1 IGDB reviewed precision.',
  'metric': 'rank_1_reviewed_precision',
  'section': 'IGDB',
 

In [33]:
read_csv(READINESS_DIR / 'ml_defense_demo_sequence.csv')


[{'evidence': 'ml_research_defense_summary.json',
  'section': 'Data pipeline',
  'show': 'source counts, candidate pairs, manual labels',
  'step': '1'},
 {'evidence': 'ablation_study.csv, calibration_bins.csv',
  'section': 'ER model',
  'show': 'v3c metrics, thresholds, calibration, ablation',
  'step': '2'},
 {'evidence': 'merge_strategy_comparison.json',
  'section': 'Merge governance',
  'show': 'trusted vs model-only vs hybrid merge strategies',
  'step': '3'},
 {'evidence': 'graph_analysis_summary.json',
  'section': 'Graph risk',
  'show': 'same-source duplicate links and risky components',
  'step': '4'},
 {'evidence': 'embedding_model_comparison.csv',
  'section': 'Embeddings',
  'show': 'TF-IDF/SVD title-vector comparison',
  'step': '5'},
 {'evidence': 'igdb_matching_summary.json',
  'section': 'IGDB',
  'show': 'rank-1 vs lower-rank retrieval quality and enrichment coverage',
  'step': '6'},
 {'evidence': 'recommendation_examples.csv',
  'section': 'Recommendations',
  's

In [34]:
read_csv(READINESS_DIR / 'ml_defense_artifact_checklist.csv')


[{'artifact_path': 'data/artifacts/reports/entity_resolution/merge_strategies/merge_strategy_comparison.json',
  'exists': 'True',
  'non_empty': 'True',
  'purpose': 'Shadow merge strategy metrics.',
  'required': 'True',
  'section': 'ER',
  'size_bytes': '4153',
  'status': 'ok'},
 {'artifact_path': 'data/artifacts/reports/ml_research_defense/ml_research_defense_summary.json',
  'exists': 'True',
  'non_empty': 'True',
  'purpose': 'Main ER research summary.',
  'required': 'True',
  'section': 'ER',
  'size_bytes': '49321',
  'status': 'ok'},
 {'artifact_path': 'data/artifacts/reports/ml_research_defense/ablation_study.csv',
  'exists': 'True',
  'non_empty': 'True',
  'purpose': 'Feature ablation study.',
  'required': 'True',
  'section': 'ER',
  'size_bytes': '1076',
  'status': 'ok'},
 {'artifact_path': 'data/artifacts/reports/ml_research_defense/calibration_bins.csv',
  'exists': 'True',
  'non_empty': 'True',
  'purpose': 'Calibration bins.',
  'required': 'True',
  'section'

## 14. Выводы

- Entity Resolution — самый сильный текущий ML-исследовательский блок.
- Ручные метки критичны: они дают и положительные, и отрицательные примеры.
- `model_auto_090` полезен для кандидатов с высокой уверенностью, но не как слепой canonical merge.
- `model_auto_070_research` полезен для анализа рисков, потому что выявляет ложноположительные совпадения и проблемы кластеров.
- Анализ графа рисков показывает транзитивный ER-риск, который не виден только по метрикам отдельных пар.
- Лёгкие эмбеддинги названий улучшают fuzzy-only baseline без скачивания внешних моделей.
- Анализ IGDB rank показывает, почему поисковые кандидаты требуют governance-контроля retrieval и выборочной ручной проверки.
- Рекомендации — объяснимый content-based baseline, его стоит показывать как вторичный ML-блок.
- Grounded RAG-like explanations только визуализируют рассчитанные факты; они не принимают решения по объединениям или рекомендациям.
- Bayesian rating демонстрирует отдельный интерпретируемый статистический блок для устойчивого ранжирования при sparse vote counts.